# 04A - SIMCA Grid Search

This notebook generates the exhaustive SIMCA grid-search candidate set for the new workflow. It uses only the PCA preprocessing shortlist from notebook 03, scoped by matrix family.

It does not select final models. It writes validation metrics and model candidate configurations for notebook 04C.

## Inputs and outputs

Inputs:
- `results/03_pca_<RESULTS_TAG>/pca_selected_preprocessings.parquet`
- `HSI Data/processed/nir_uco_database.h5`

Outputs:
- `grid_search_results.parquet`: validation 2-way metrics for all grid candidates
- `grid_model_candidates.parquet`: deduplicated model candidate configs for 04C
- `grid_search_errors.parquet`: failed grid configurations
- `grid_search_diagnostics.parquet`: compact monitoring table
- `grid_search_protocol.parquet`: run configuration

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.utils import load_parquet, save_parquet, list_result_files
from src.workflows.simca_tables import compact_simca_table_for_path
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.workflows.simca import (
    make_target_train_filters,
    run_simca_rule_variant_grid,
)
from src.workflows.simca_candidates import (
    add_selection_track,
    add_simca_candidate_ids,
    build_pca_preprocessing_configs_by_matrix_family,
    deduplicate_simca_candidates,
    filter_simca_candidates_by_pca_preprocessing,
    validate_simca_candidate_contract,
    validate_simca_evaluation_contract,
)
from src.workflows.simca_selection_utils import materialize_selection_metrics

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

The grid keeps object and pixel matrix families separate. PCA-selected object preprocessings are applied only to object matrices; PCA-selected pixel preprocessings are applied only to pixel matrices.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE
RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"04A_simca_grid_search_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"

GRID_RESULTS_PATH = RESULTS_DIR / "grid_search_results.parquet"
GRID_MODEL_CANDIDATES_PATH = RESULTS_DIR / "grid_model_candidates.parquet"
GRID_ERRORS_PATH = RESULTS_DIR / "grid_search_errors.parquet"
GRID_DIAGNOSTICS_PATH = RESULTS_DIR / "grid_search_diagnostics.parquet"
GRID_PROTOCOL_PATH = RESULTS_DIR / "grid_search_protocol.parquet"
WAVELENGTH_CONFIG_PATH = RESULTS_DIR / "wavelength_config.parquet"
PREPROCESSING_SCOPE_PATH = RESULTS_DIR / "preprocessing_scope.parquet"

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = list(expcfg.REFERENCE_CLASSES)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=expcfg.SIMCA_TRAIN_BATCHES,
)
VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": REFERENCE_CLASSES,
    "batch": list(expcfg.SIMCA_VALIDATION_BATCHES),
}

MATRIX_METHODS = ["object_mean", "object_median", "balanced_pixels"]
RULE_VARIANTS = [
    "simple_chi2",
    "simple_emp_cv",
    "alternative_chi2_fixed2",
    "alternative_chi2_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_empHQ_emp_cv",
    "data_driven_chi2",
    "data_driven_emp_cv",
    "combined_index_chi2",
]

N_COMPONENTS_VALUES = [3, 4, 5, 6, 7, 8, 10, 11, 12]
ALPHA_VALUES = list(expcfg.SIMCA_ALPHA_VALUES)
OBJECT_THRESHOLDS = list(expcfg.SIMCA_OBJECT_THRESHOLDS)

M_VALUES = [expcfg.M_BALANCED_PIXELS]
BALANCED_PIXEL_STRATEGY_VALUES = list(expcfg.BALANCED_PIXEL_STRATEGIES)
SG_WINDOW_LENGTH_VALUES = [11]
SG_POLYORDER_VALUES = [2]
POSITION_DILATION_RADIUS_VALUES = [3]

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

RUN_GRID_SEARCH = True
RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

track_specs_df = pd.DataFrame([
    {"selection_track": track, **spec}
    for track, spec in expcfg.SIMCA_SELECTION_TRACK_SPECS.items()
])

display(track_specs_df)
print("DB_H5_PATH:", DB_H5_PATH)
print("PCA_SELECTED_PREPROCESSINGS_PATH:", PCA_SELECTED_PREPROCESSINGS_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

,selection_track,matrix_family,decision_mode,primary_metric_level,secondary_metric_level
0,object_matrix_2way,object_matrix,2way,object,pixel
1,object_matrix_3way,object_matrix,3way,object,pixel
2,pixel_matrix_2way,pixel_matrix,2way,pixel,object
3,pixel_matrix_3way,pixel_matrix,3way,pixel,object


DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
PCA_SELECTED_PREPROCESSINGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all


## Load inputs

Notebook 03 is the only source of preprocessing preselection. The family-specific mapping built here is passed directly to the SIMCA grid.

In [3]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")
if not PCA_SELECTED_PREPROCESSINGS_PATH.exists():
    raise FileNotFoundError(
        f"PCA shortlist not found: {PCA_SELECTED_PREPROCESSINGS_PATH}. Run notebook 03 first."
    )

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
    pca_selected_preprocessings_df
)

preprocessing_scope_df = pd.DataFrame([
    {
        "matrix_family": family,
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
    }
    for family, configs in preprocessing_configs_by_family.items()
    for name, steps in configs.items()
])

if preprocessing_scope_df.empty:
    raise RuntimeError("The PCA preprocessing shortlist is empty.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

save_parquet(compact_simca_table_for_path(wavelength_config_df, WAVELENGTH_CONFIG_PATH), WAVELENGTH_CONFIG_PATH)
save_parquet(compact_simca_table_for_path(preprocessing_scope_df, PREPROCESSING_SCOPE_PATH), PREPROCESSING_SCOPE_PATH)

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
display(wavelength_config_df)
display(preprocessing_scope_df.sort_values(["matrix_family", "preprocessing"]))

Number of images: 48
Number of objects: 1262


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,matrix_family,preprocessing,preprocessing_steps
0,object_matrix,absorbance_sg_d1,absorbance+sg_d1
1,object_matrix,absorbance_sg_d2,absorbance+sg_d2
4,object_matrix,absorbance_snv_sg_d1,absorbance+snv+sg_d1
2,object_matrix,absorbance_snv_sg_d2,absorbance+snv+sg_d2
3,object_matrix,snv_sg_d2,snv+sg_d2
6,pixel_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth
9,pixel_matrix,raw,raw
8,pixel_matrix,sg_smooth,sg_smooth
7,pixel_matrix,snv,snv
5,pixel_matrix,snv_sg_smooth,snv+sg_smooth


## Run grid search

The output metric table is explicitly tagged as 2-way object-level validation. Three-way decision thresholds are calibrated later from the same model candidates.

In [4]:
if RUN_GRID_SEARCH:
    grid_results_df, _grid_result_store, grid_errors_df = run_simca_rule_variant_grid(
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=preprocessing_configs_by_family,
        matrix_methods=MATRIX_METHODS,
        rule_variants=RULE_VARIANTS,
        n_components_values=N_COMPONENTS_VALUES,
        alpha_values=ALPHA_VALUES,
        object_thresholds=OBJECT_THRESHOLDS,
        m_values=M_VALUES,
        random_state=RANDOM_STATE,
        replace=REPLACE_BALANCED_PIXELS,
        wavelengths=wavelengths,
        sg_window_length_values=SG_WINDOW_LENGTH_VALUES,
        sg_polyorder_values=SG_POLYORDER_VALUES,
        position_dilation_radius_values=POSITION_DILATION_RADIUS_VALUES,
        cv_n_splits=CV_N_SPLITS,
        group_col=CV_GROUP_COL,
        keep_pixel_tables=False,
        keep_cv_tables=False,
        verbose=True,
        balanced_pixel_strategy_values=BALANCED_PIXEL_STRATEGY_VALUES,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
else:
    grid_results_df = pd.DataFrame()
    grid_errors_df = pd.DataFrame()

if grid_results_df.empty:
    raise RuntimeError("The grid search produced no result rows.")

grid_results_df = materialize_selection_metrics(grid_results_df)
grid_results_df = add_simca_candidate_ids(
    grid_results_df.assign(candidate_source="04A_grid_search")
)
grid_results_df["model_candidate_id"] = grid_results_df["candidate_id"]

grid_metrics_df = grid_results_df.copy()
grid_metrics_df["decision_mode"] = "2way"
grid_metrics_df["evaluation_stage"] = "validation_batch_3"
grid_metrics_df["metric_level"] = "object"
grid_metrics_df = add_selection_track(grid_metrics_df)
validate_simca_evaluation_contract(grid_metrics_df)

grid_model_candidates_df = deduplicate_simca_candidates(
    grid_results_df.assign(candidate_source="04A_grid_search")
)
grid_model_candidates_df = filter_simca_candidates_by_pca_preprocessing(
    grid_model_candidates_df,
    pca_selected_preprocessings_df,
    strict=True,
)
grid_model_candidates_df["model_candidate_id"] = grid_model_candidates_df["candidate_id"]

sort_cols = [
    col for col in [
        "matrix_family",
        "matrix_method",
        "fn_rate",
        "fp_rate",
        "balanced_accuracy",
        "preprocessing",
        "rule_variant",
        "n_components",
        "object_threshold",
    ]
    if col in grid_model_candidates_df.columns
]
ascending = [
    False if col == "balanced_accuracy" else True
    for col in sort_cols
]
grid_model_candidates_df = (
    grid_model_candidates_df
    .sort_values(sort_cols, ascending=ascending)
    .reset_index(drop=True)
)
grid_model_candidates_df["selected_config_id"] = [
    f"04A_grid_{i:06d}" for i in range(len(grid_model_candidates_df))
]
validate_simca_candidate_contract(grid_model_candidates_df)

grid_metrics_df = grid_metrics_df.merge(
    grid_model_candidates_df[["candidate_id", "selected_config_id"]],
    on="candidate_id",
    how="left",
    validate="many_to_one",
)

display(grid_metrics_df.head())
display(grid_model_candidates_df.head())


[1/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=3 | alpha=0.01 | SG=(11,2) | dilation=3

[2/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=4 | alpha=0.01 | SG=(11,2) | dilation=3

[3/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=5 | alpha=0.01 | SG=(11,2) | dilation=3

[4/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=6 | alpha=0.01 | SG=(11,2) | dilation=3

[5/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=7 | alpha=0.01 | SG=(11,2) | dilation=3

[6/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=8 | alpha=0.01 | SG=(11,2) | dilation=3

[7/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=10 | alpha=0.01 | SG=(11,2) | dilation=3

[8/180] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=11 | alpha=0.01 | SG=(11,2) | dilation=3

[9/180] empirical_cv | matrix=object_

,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,...,cv_rule_limit,candidate_source,selected_rule_name,candidate_id,model_candidate_id,decision_mode,evaluation_stage,metric_level,selection_track,selected_config_id
0,peanut,almond,108,53,0,30,25,1.0,0.454545,0.727273,...,2.0,04A_grid_search,alternative_chi2_fixed2,simca_d6ba40560e235148,simca_d6ba40560e235148,2way,validation_batch_3,object,pixel_matrix_2way,04A_grid_001620
1,peanut,almond,108,53,0,31,24,1.0,0.436364,0.718182,...,NaN,04A_grid_search,data_driven_chi2,simca_1b78e80584a64e2e,simca_1b78e80584a64e2e,2way,validation_batch_3,object,pixel_matrix_2way,04A_grid_001621
2,peanut,almond,108,53,0,36,19,1.0,0.345455,0.672727,...,2.0,04A_grid_search,alternative_chi2_fixed2,simca_3666f161569a37d5,simca_3666f161569a37d5,2way,validation_batch_3,object,pixel_matrix_2way,04A_grid_001622
3,peanut,almond,108,53,0,38,17,1.0,0.309091,0.654545,...,NaN,04A_grid_search,data_driven_chi2,simca_9876de5fbfea187a,simca_9876de5fbfea187a,2way,validation_batch_3,object,pixel_matrix_2way,04A_grid_001623
4,peanut,almond,108,53,0,38,17,1.0,0.309091,0.654545,...,1.0,04A_grid_search,simple_chi2,simca_a97a347d2881e6f9,simca_a97a347d2881e6f9,2way,validation_batch_3,object,pixel_matrix_2way,04A_grid_001624


,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,...,cv_abs_rejection_error,cv_rule_limit,candidate_source,selected_rule_name,candidate_id,model_candidate_id,candidate_sources,n_candidate_sources,n_duplicate_rows,selected_config_id
0,peanut,almond,108,42,11,33,22,0.792453,0.400000,0.596226,...,0.000204,33.805794,04A_grid_search,simple_emp_cv,simca_a09c9451a34953dd,simca_a09c9451a34953dd,04A_grid_search,1,1,04A_grid_000000
1,peanut,almond,108,39,14,29,26,0.735849,0.472727,0.604288,...,0.000204,2784.636902,04A_grid_search,data_driven_emp_cv,simca_d0f627c9cc0d8578,simca_d0f627c9cc0d8578,04A_grid_search,1,1,04A_grid_000001
2,peanut,almond,108,34,19,25,30,0.641509,0.545455,0.593482,...,0.000204,33.805794,04A_grid_search,simple_emp_cv,simca_73a0fccdc6679467,simca_73a0fccdc6679467,04A_grid_search,1,1,04A_grid_000002
3,peanut,almond,108,33,20,19,36,0.622642,0.654545,0.638593,...,0.000204,2784.636902,04A_grid_search,data_driven_emp_cv,simca_e9a573f6658e308f,simca_e9a573f6658e308f,04A_grid_search,1,1,04A_grid_000003
4,peanut,almond,108,29,24,15,40,0.547170,0.727273,0.637221,...,0.000204,20.984841,04A_grid_search,simple_emp_cv,simca_0cf8448886f162c6,simca_0cf8448886f162c6,04A_grid_search,1,1,04A_grid_000004


## Diagnostics and save

In [5]:
grid_diagnostics_df = (
    grid_metrics_df
    .groupby(
        [
            "selection_track",
            "matrix_family",
            "matrix_method",
            "preprocessing",
            "rule_variant",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("candidate_id", "size"),
        n_candidates=("candidate_id", "nunique"),
        best_fn_rate=("fn_rate", "min"),
        best_fp_rate=("fp_rate", "min"),
        best_balanced_accuracy=("balanced_accuracy", "max"),
    )
    .reset_index()
    .sort_values(
        ["matrix_family", "best_fn_rate", "best_fp_rate", "best_balanced_accuracy"],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)

grid_protocol_df = pd.DataFrame([{
    "notebook": "04A_simca_grid_search",
    "results_tag": RESULTS_TAG,
    "db_h5_path": str(DB_H5_PATH),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),
    "matrix_methods_json": json.dumps(MATRIX_METHODS),
    "rule_variants_json": json.dumps(RULE_VARIANTS),
    "n_components_values_json": json.dumps(N_COMPONENTS_VALUES),
    "alpha_values_json": json.dumps(ALPHA_VALUES),
    "object_thresholds_json": json.dumps(OBJECT_THRESHOLDS),
    "m_values_json": json.dumps(M_VALUES),
    "balanced_pixel_strategy_values_json": json.dumps(BALANCED_PIXEL_STRATEGY_VALUES),
    "sg_window_length_values_json": json.dumps(SG_WINDOW_LENGTH_VALUES),
    "sg_polyorder_values_json": json.dumps(SG_POLYORDER_VALUES),
    "position_dilation_radius_values_json": json.dumps(POSITION_DILATION_RADIUS_VALUES),
    "generated_decision_modes_json": json.dumps(["2way"]),
    "downstream_decision_modes_json": json.dumps(list(expcfg.SIMCA_DECISION_MODES)),
    "n_grid_metric_rows": int(len(grid_metrics_df)),
    "n_grid_model_candidates": int(len(grid_model_candidates_df)),
    "n_grid_errors": int(len(grid_errors_df)),
    "grid_results_path": str(GRID_RESULTS_PATH),
    "grid_model_candidates_path": str(GRID_MODEL_CANDIDATES_PATH),
}])

save_parquet(compact_simca_table_for_path(grid_metrics_df, GRID_RESULTS_PATH), GRID_RESULTS_PATH)
save_parquet(compact_simca_table_for_path(grid_model_candidates_df, GRID_MODEL_CANDIDATES_PATH), GRID_MODEL_CANDIDATES_PATH)
save_parquet(compact_simca_table_for_path(grid_errors_df, GRID_ERRORS_PATH), GRID_ERRORS_PATH)
save_parquet(compact_simca_table_for_path(grid_diagnostics_df, GRID_DIAGNOSTICS_PATH), GRID_DIAGNOSTICS_PATH)
save_parquet(compact_simca_table_for_path(grid_protocol_df, GRID_PROTOCOL_PATH), GRID_PROTOCOL_PATH)

print("Saved:")
for path in [
    GRID_RESULTS_PATH,
    GRID_MODEL_CANDIDATES_PATH,
    GRID_ERRORS_PATH,
    GRID_DIAGNOSTICS_PATH,
    GRID_PROTOCOL_PATH,
]:
    print(" -", path)

display(grid_diagnostics_df.head(20))
display(list_result_files(RESULTS_DIR).head(20))

Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_search_results.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_model_candidates.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_search_errors.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_search_diagnostics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_non_noisy_all\grid_search_protocol.parquet


,selection_track,matrix_family,matrix_method,preprocessing,rule_variant,n_rows,n_candidates,best_fn_rate,best_fp_rate,best_balanced_accuracy
0,object_matrix_2way,object_matrix,object_median,absorbance_sg_d1,simple_emp_cv,18,18,0.037736,0.036364,0.691767
1,object_matrix_2way,object_matrix,object_median,absorbance_sg_d1,data_driven_emp_cv,18,18,0.056604,0.036364,0.691767
2,object_matrix_2way,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,18,18,0.056604,0.054545,0.647684
3,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d1,data_driven_emp_cv,18,18,0.056604,0.363636,0.556775
4,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d1,simple_emp_cv,18,18,0.056604,0.509091,0.594511
5,object_matrix_2way,object_matrix,object_median,snv_sg_d2,simple_emp_cv,18,18,0.056604,0.654545,0.531218
6,object_matrix_2way,object_matrix,object_median,snv_sg_d2,data_driven_emp_cv,18,18,0.056604,0.672727,0.533962
7,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d2,data_driven_emp_cv,18,18,0.075472,0.454545,0.575643
8,object_matrix_2way,object_matrix,object_median,absorbance_snv_sg_d1,alternative_chi2_emp_cv,18,18,0.094340,0.381818,0.601544
9,object_matrix_2way,object_matrix,object_median,snv_sg_d2,alternative_chi2_emp_cv,18,18,0.094340,0.381818,0.554374


,file,suffixes,size_mb
0,grid_search_results.parquet,.parquet,0.124867
1,grid_model_candidates.parquet,.parquet,0.119121
2,grid_search_protocol.parquet,.parquet,0.018837
3,grid_search_diagnostics.parquet,.parquet,0.008195
4,wavelength_config.parquet,.parquet,0.003999
5,preprocessing_scope.parquet,.parquet,0.002533
6,grid_search_errors.parquet,.parquet,0.000607
